# Week 9: The Platinum Handcuffs — Measuring Switching Costs
## Stratified Cox & Composite Lock-In Index

**Masterclass:** [Week 9 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week9_platinum_handcuffs.html)

**Dataset:** IBM Telco Customer Churn

**What You'll Build:**
1. Switching cost proxy variables from Telco data
2. Stratified Cox by contract type
3. Composite switching cost index
4. Lock-in vs. loyalty analysis

In [ ]:
!pip install lifelines pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter, KaplanMeierFitter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df['churned'] = (df['Churn'] == 'Yes').astype(int)
print(f"Customers: {len(df):,}")

---
## Part 1: Build Switching Cost Proxies

We can't directly observe switching costs, but we can proxy them using features that represent investment in the current provider.

In [ ]:
# Switching cost proxies
df['n_services'] = (
    (df['PhoneService'] == 'Yes').astype(int) +
    (df['InternetService'] != 'No').astype(int) +
    (df['OnlineSecurity'] == 'Yes').astype(int) +
    (df['OnlineBackup'] == 'Yes').astype(int) +
    (df['DeviceProtection'] == 'Yes').astype(int) +
    (df['TechSupport'] == 'Yes').astype(int) +
    (df['StreamingTV'] == 'Yes').astype(int) +
    (df['StreamingMovies'] == 'Yes').astype(int)
)
df['has_long_contract'] = (df['Contract'] != 'Month-to-month').astype(int)
df['auto_pay'] = df['PaymentMethod'].isin(
    ['Bank transfer (automatic)', 'Credit card (automatic)']).astype(int)

# Composite switching cost index (weighted rank)
from scipy.stats import rankdata
df['sc_services'] = rankdata(df['n_services']) / len(df)
df['sc_tenure'] = rankdata(df['tenure']) / len(df)
df['sc_contract'] = df['has_long_contract'].astype(float)
df['sc_autopay'] = df['auto_pay'].astype(float)

df['switching_cost_index'] = (
    0.30 * df['sc_services'] +
    0.25 * df['sc_tenure'] +
    0.25 * df['sc_contract'] +
    0.20 * df['sc_autopay']
)

# Churn rate by switching cost quartile
df['sc_quartile'] = pd.qcut(df['switching_cost_index'], 4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])
churn_by_sc = df.groupby('sc_quartile')['churned'].mean()
print("=== Churn Rate by Switching Cost Quartile ===")
print(churn_by_sc.apply(lambda x: f"{x:.1%}"))

In [ ]:
# KM curves by switching cost quartile
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Q1 (Low)': '#c45d3e', 'Q2': '#e9c46a', 'Q3': '#264653', 'Q4 (High)': '#2d6a4f'}
for q in ['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']:
    mask = df['sc_quartile'] == q
    kmf = KaplanMeierFitter()
    kmf.fit(df.loc[mask, 'tenure'], df.loc[mask, 'churned'], label=q)
    kmf.plot_survival_function(ax=ax, color=colors[q], linewidth=2)

ax.set_title('Survival by Switching Cost Quartile', fontsize=14, fontweight='bold')
ax.set_xlabel('Tenure (Months)')
ax.set_ylabel('Survival Probability')
ax.legend(title='Switching Cost')
plt.tight_layout()
plt.show()

---
## Part 2: Stratified Cox by Contract Type

In [ ]:
covs = ['n_services', 'auto_pay', 'SeniorCitizen', 'MonthlyCharges']

# Stratified Cox: each contract type gets its own baseline hazard
cph = CoxPHFitter(penalizer=0.01)
cph.fit(df[covs + ['tenure', 'churned', 'Contract']],
        duration_col='tenure', event_col='churned', strata=['Contract'])

cph.print_summary()

print("\n=== Key Insight ===")
hr_services = np.exp(cph.params_['n_services'])
print(f"Each additional service reduces churn hazard by {(1-hr_services)*100:.0f}%")
print("This is product-driven lock-in: more services = harder to leave")

---
## Exercises

### Exercise 1: Lock-In vs Loyalty
Customers with high switching costs and high satisfaction = loyal. High switching costs and low satisfaction = locked-in (hostage). Use `TechSupport` and `OnlineSecurity` as satisfaction proxies. Segment the base into loyal vs. locked-in.

### Exercise 2: Optimal Service Bundle
Which combination of services minimizes churn? Use the Cox model to find the bundle with the lowest predicted hazard.

### Exercise 3: Contract Hazard Curve
Plot the baseline hazard for each contract strata. Do you see the "bathtub curve" at renewal windows?

---
## Key Takeaways
1. **Switching costs** are measurable via proxy variables (services, tenure, contract, autopay)
2. **Product-driven lock-in** (more services) is more durable than contractual lock-in
3. **Stratified Cox** separates contract effects from covariate effects
4. **Lock-in ≠ loyalty** — distinguish the two to prevent hostage churn